# Partition readers of permutations — computational companion

Dean Kulik, QuHarmonics Research Group (ORCID 0009-0003-3128-8828)

This notebook is self-contained: it uses only the Python standard library, reads no files, and reproduces every computed number in the paper. Run the cells top to bottom; the full run takes about a minute.

**Conventions.** The field has $N$ cells $0,\dots,N-1$. A permutation is a tuple `p` with `p[x]` the image of `x`. A reader at depth $m$ (with $m$ dividing $N$) cuts the field into $m$ contiguous blocks of size $s=N/m$ and returns the table $C_m(p)_{ij}$ = number of cells of block $i$ sent into block $j$.

In [1]:
import itertools, hashlib, math, random, time
from collections import Counter, defaultdict
from math import lgamma, log, log2, pi, e, comb, factorial

LN2 = log(2)

def log2_factorial(k):
    return lgamma(k + 1) / LN2

def table(p, m):
    """Reader at depth m: m contiguous blocks of equal size, returns the block-to-block crossing table."""
    n = len(p); s = n // m
    T = [[0] * m for _ in range(m)]
    for x, y in enumerate(p):
        T[x // s][y // s] += 1
    return tuple(map(tuple, T))

def log2_fiber(T):
    """log2 of the number of permutations with table T: (prod of block sizes!)^2 / prod T_ij!."""
    d = [sum(r) for r in T]
    return 2 * sum(log2_factorial(x) for x in d) - sum(log2_factorial(v) for r in T for v in r)

def hyp_logpmf(N, s, k):
    """natural log of P(c = k) for c ~ Hypergeometric(population N, s marked, s drawn)."""
    lc = lambda a, b: lgamma(a + 1) - lgamma(b + 1) - lgamma(a - b + 1)
    return lc(s, k) + lc(N - s, s - k) - lc(N, s)

def profile_formula(N, m):
    """Theorem 4.1: I(T; C_m) = log2 N! - 2 m log2 s! + m^2 E[log2 c!],  c ~ Hyp(N, s, s)."""
    if m == 1:
        return 0.0
    s = N // m
    E = sum(math.exp(hyp_logpmf(N, s, k)) * log2_factorial(k) for k in range(s + 1) if hyp_logpmf(N, s, k) > -700)
    return log2_factorial(N) - 2 * m * log2_factorial(s) + m * m * E

def hyp_entropy(N, s):
    H = 0.0
    for k in range(s + 1):
        lp = hyp_logpmf(N, s, k)
        if lp > -700:
            H -= math.exp(lp) * lp / LN2
    return H

def inverse(p):
    q = [0] * len(p)
    for i, v in enumerate(p):
        q[v] = i
    return tuple(q)

def is_symmetric(T):
    return all(T[i][j] == T[j][i] for i in range(len(T)) for j in range(len(T)))

def cycle_count(p):
    seen = [False] * len(p); c = 0
    for s0 in range(len(p)):
        if not seen[s0]:
            c += 1; x = s0
            while not seen[x]:
                seen[x] = True; x = p[x]
    return c

def components(T):
    m = len(T); parent = list(range(m))
    def f(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    for i in range(m):
        for j in range(m):
            if T[i][j]:
                parent[f(i)] = f(j)
    return len({f(i) for i in range(m)})

def divisors(n):
    return [m for m in range(1, n + 1) if n % m == 0]
print("definitions loaded")

definitions loaded


## Lemma 3.1 and Proposition 3.3: fiber sizes and nesting (exhaustive for $N=8$ and $N=9$)

In [2]:
for N in (8, 9):
    perms = list(itertools.permutations(range(N)))
    for m in divisors(N):
        fibers = Counter(table(p, m) for p in perms)
        bad = sum(abs(log2(c) - log2_fiber(T)) > 1e-9 for T, c in fibers.items())
        print(f"N={N} m={m}: {len(fibers):>6} distinct tables, fiber-size mismatches {bad}")
    # nesting: readers whose blocks refine each other have nested fibers
    for m1, m2 in [(a, b) for a in divisors(N) for b in divisors(N) if b % a == 0 and a < b]:
        coarse_of = {}
        bad = 0
        for p in perms:
            fine, coarse = table(p, m2), table(p, m1)
            if coarse_of.setdefault(fine, coarse) != coarse:
                bad += 1
        print(f"   refinement m={m1} -> m={m2}: finer table determines coarser table, violations {bad}")

N=8 m=1:      1 distinct tables, fiber-size mismatches 0
N=8 m=2:      5 distinct tables, fiber-size mismatches 0


N=8 m=4:    282 distinct tables, fiber-size mismatches 0


N=8 m=8:  40320 distinct tables, fiber-size mismatches 0
   refinement m=1 -> m=2: finer table determines coarser table, violations 0
   refinement m=1 -> m=4: finer table determines coarser table, violations 0


   refinement m=1 -> m=8: finer table determines coarser table, violations 0
   refinement m=2 -> m=4: finer table determines coarser table, violations 0


   refinement m=2 -> m=8: finer table determines coarser table, violations 0


   refinement m=4 -> m=8: finer table determines coarser table, violations 0


... (2 outputs left out)

N=9 m=9: 362880 distinct tables, fiber-size mismatches 0


   refinement m=1 -> m=3: finer table determines coarser table, violations 0


   refinement m=1 -> m=9: finer table determines coarser table, violations 0


   refinement m=3 -> m=9: finer table determines coarser table, violations 0


## Theorem 4.1: exact information profile (enumeration for $N=8,9$; sampling for $N=128$)

In [3]:
for N in (8, 9):
    perms = list(itertools.permutations(range(N)))
    for m in divisors(N):
        cnt = Counter(table(p, m) for p in perms)
        H = -sum(c / len(perms) * log2(c / len(perms)) for c in cnt.values())
        print(f"N={N} m={m}: exact H(C_m) = {H:.6f}   formula = {profile_formula(N, m):.6f}")

N = 128
rng = random.Random(20260911)
sample = []
for _ in range(2000):
    p = list(range(N)); rng.shuffle(p); sample.append(tuple(p))
total = log2_factorial(N)
print(f"\nN=128, log2(128!) = {total:.3f}; 2000 uniform permutations (seed 20260911)")
for m in (1, 2, 4, 8, 16, 32, 64, 128):
    mean_seen = sum(total - log2_fiber(table(p, m)) for p in sample) / len(sample)
    print(f"  m={m:>3}: formula {profile_formula(N, m):9.3f}   sample mean {mean_seen:9.3f}")

N=8 m=1: exact H(C_m) = -0.000000   formula = 0.000000
N=8 m=2: exact H(C_m) = 1.641893   formula = 1.641893
N=8 m=4: exact H(C_m) = 7.870637   formula = 7.870637


N=8 m=8: exact H(C_m) = 15.299208   formula = 15.299208


N=9 m=1: exact H(C_m) = -0.000000   formula = 0.000000


N=9 m=3: exact H(C_m) = 5.164890   formula = 5.164890


N=9 m=9: exact H(C_m) = 18.469133   formula = 18.469133

N=128, log2(128!) = 716.162; 2000 uniform permutations (seed 20260911)
  m=  1: formula     0.000   sample mean     0.000
  m=  2: formula     3.553   sample mean     3.551
  m=  4: formula    25.936   sample mean    25.980
  m=  8: formula   101.325   sample mean   101.331
  m= 16: formula   250.332   sample mean   250.336


  m= 32: formula   427.240   sample mean   427.229


  m= 64: formula   588.666   sample mean   588.675


  m=128: formula   716.162   sample mean   716.162


## Corollary 4.3: the first reader ($m=2$)

In [4]:
print(" N         exact H(C_2)   0.5*log2(pi*e*N/8)")
for N in (8, 64, 128, 1024, 65536, 2 ** 20):
    print(f" {N:<9} {hyp_entropy(N, N // 2):12.4f}   {0.5 * log2(pi * e * N / 8):12.4f}")

 N         exact H(C_2)   0.5*log2(pi*e*N/8)
 8               1.6419         1.5471
 64              3.0584         3.0471
 128             3.5527         3.5471
 1024            5.0478         5.0471
 65536           8.0471         8.0471


 1048576        10.0471        10.0471


## Section 5: direction (exact for small $N$, sampled for $N=128$)

In [5]:
def involutions(n):
    a, b = 1, 1
    for k in range(2, n + 1):
        a, b = b, b + (k - 1) * a
    return b if n >= 1 else 1

for N, m in [(4, 2), (4, 4), (6, 2), (6, 3), (6, 6), (8, 2), (8, 4), (8, 8), (9, 3), (9, 9)]:
    perms = list(itertools.permutations(range(N)))
    k = 0; bad = 0
    for p in perms:
        T = table(p, m)
        k += is_symmetric(T)
        bad += (table(inverse(p), m) != tuple(zip(*T)))        # C(p^-1) must equal C(p) transposed
    extra = f"   (involutions of {N}: {involutions(N)})" if m == N else ""
    print(f"N={N} m={m}: blind to direction on {k}/{len(perms)} = {k / len(perms):.4f}; transpose-law violations {bad}{extra}")

N = 128
rng = random.Random(7)
sample = []
for _ in range(20000):
    p = list(range(N)); rng.shuffle(p); sample.append(p)
print("\nN=128, 20000 uniform permutations (seed 7)")
for m in (2, 4, 8, 16, 32, 64, 128):
    k = sum(is_symmetric(table(p, m)) for p in sample)
    print(f"  m={m:>3}: fraction blind to direction {k / len(sample):.4f}")

N=4 m=2: blind to direction on 24/24 = 1.0000; transpose-law violations 0
N=4 m=4: blind to direction on 10/24 = 0.4167; transpose-law violations 0   (involutions of 4: 10)
N=6 m=2: blind to direction on 720/720 = 1.0000; transpose-law violations 0
N=6 m=3: blind to direction on 384/720 = 0.5333; transpose-law violations 0
N=6 m=6: blind to direction on 76/720 = 0.1056; transpose-law violations 0   (involutions of 6: 76)
N=8 m=2: blind to direction on 40320/40320 = 1.0000; transpose-law violations 0


N=8 m=4: blind to direction on 7584/40320 = 0.1881; transpose-law violations 0


N=8 m=8: blind to direction on 764/40320 = 0.0189; transpose-law violations 0   (involutions of 8: 764)


N=9 m=3: blind to direction on 164160/362880 = 0.4524; transpose-law violations 0


N=9 m=9: blind to direction on 2620/362880 = 0.0072; transpose-law violations 0   (involutions of 9: 2620)



N=128, 20000 uniform permutations (seed 7)
  m=  2: fraction blind to direction 1.0000


  m=  4: fraction blind to direction 0.0037
  m=  8: fraction blind to direction 0.0000


  m= 16: fraction blind to direction 0.0000


  m= 32: fraction blind to direction 0.0000


  m= 64: fraction blind to direction 0.0000


  m=128: fraction blind to direction 0.0000


## Theorem 6.1 and Proposition 6.2: loops (exhaustive for $N=8$ and $N=9$)

In [6]:
for N in (8, 9):
    perms = list(itertools.permutations(range(N)))
    Hn = sum(1 / k for k in range(1, N + 1))
    for m in divisors(N):
        fib = defaultdict(set); mincyc = {}
        for p in perms:
            T = table(p, m); c = cycle_count(p)
            fib[T].add(c); mincyc[T] = min(mincyc.get(T, N), c)
        fixed = sum(len(v) == 1 for v in fib.values())
        comp_bad = sum(mincyc[T] != components(T) for T in fib)
        consecutive = all(set(range(min(v), max(v) + 1)) == v for v in fib.values())
        mean_cert = sum(components(table(p, m)) for p in perms) / len(perms)
        print(f"N={N} m={m}: tables fixing the number of loops {fixed}/{len(fib)} | min loops = components violations {comp_bad} | "
              f"loop counts in each fiber form an interval: {consecutive} | mean certified loops {mean_cert:.4f} (mean actual {Hn:.4f})")

N=8 m=1: tables fixing the number of loops 0/1 | min loops = components violations 0 | loop counts in each fiber form an interval: True | mean certified loops 1.0000 (mean actual 2.7179)


N=8 m=2: tables fixing the number of loops 0/5 | min loops = components violations 0 | loop counts in each fiber form an interval: True | mean certified loops 1.0143 (mean actual 2.7179)


N=8 m=4: tables fixing the number of loops 0/282 | min loops = components violations 0 | loop counts in each fiber form an interval: True | mean certified loops 1.1722 (mean actual 2.7179)


N=8 m=8: tables fixing the number of loops 40320/40320 | min loops = components violations 0 | loop counts in each fiber form an interval: True | mean certified loops 2.7179 (mean actual 2.7179)


N=9 m=1: tables fixing the number of loops 0/1 | min loops = components violations 0 | loop counts in each fiber form an interval: True | mean certified loops 1.0000 (mean actual 2.8290)


N=9 m=3: tables fixing the number of loops 0/55 | min loops = components violations 0 | loop counts in each fiber form an interval: True | mean certified loops 1.0351 (mean actual 2.8290)


N=9 m=9: tables fixing the number of loops 362880/362880 | min loops = components violations 0 | loop counts in each fiber form an interval: True | mean certified loops 2.8290 (mean actual 2.8290)


## Section 7: unbalanced transitions — minimum erasure (exhaustive over all maps, $n\\le 6$)

In [7]:
def partitions(n, maxpart=None):
    if maxpart is None: maxpart = n
    if n == 0:
        yield (); return
    for k in range(min(n, maxpart), 0, -1):
        for rest in partitions(n - k, k):
            yield (k,) + rest

def xlog2x(x):
    return x * log2(x) if x > 1 else 0.0

def phi(c, d):
    if c <= d:
        return 0.0
    q, r = divmod(c, d)
    return (d - r) * xlog2x(q) + r * xlog2x(q + 1)

def recurrent_points(f):
    n = len(f); on = set()
    for s0 in range(n):
        x = s0
        for _ in range(n):
            x = f[x]
        y = x
        while True:
            on.add(y); y = f[y]
            if y == x:
                break
    return len(on)

t0 = time.time()
for n in range(1, 7):
    maps = tabs = bad_min = bad_zero = bad_2D = bad_rec = 0
    for mu in partitions(n):
        b = []
        for i, d in enumerate(mu):
            b += [i] * d
        m = len(mu); best = {}
        for f in itertools.product(range(n), repeat=n):
            maps += 1
            T = [[0] * m for _ in range(m)]
            for x, y in enumerate(f):
                T[b[x]][b[y]] += 1
            T = tuple(map(tuple, T))
            E = sum(xlog2x(v) for v in Counter(f).values()) / n
            c = [sum(T[i][j] for i in range(m)) for j in range(m)]
            D = sum(max(0, c[j] - mu[j]) for j in range(m))
            bad_2D += E + 1e-12 < 2 * D / n
            bad_rec += recurrent_points(f) > n - D
            if T not in best or E < best[T][0]:
                best[T] = (E, c)
        for T, (Emin, c) in best.items():
            tabs += 1
            bad_min += abs(Emin - sum(phi(c[j], mu[j]) for j in range(m)) / n) > 1e-9
            bad_zero += (abs(Emin) < 1e-12) != (list(c) == list(mu))
    print(f"n={n}: {maps:>7,} maps, {tabs:>6,} tables | minimum-formula mismatches {bad_min} | zero iff balanced mismatches {bad_zero} | "
          f"2D/n violations {bad_2D} | recurrent-point bound violations {bad_rec}")
print(f"[{time.time() - t0:.0f}s]")
k_B, T_room = 1.380649e-23, 300.0
print(f"Landauer bound at 300 K: {k_B * T_room * log(2):.3e} J per bit")

n=1:       1 maps,      1 tables | minimum-formula mismatches 0 | zero iff balanced mismatches 0 | 2D/n violations 0 | recurrent-point bound violations 0
n=2:       8 maps,      5 tables | minimum-formula mismatches 0 | zero iff balanced mismatches 0 | 2D/n violations 0 | recurrent-point bound violations 0
n=3:      81 maps,     34 tables | minimum-formula mismatches 0 | zero iff balanced mismatches 0 | 2D/n violations 0 | recurrent-point bound violations 0
n=4:   1,280 maps,    328 tables | minimum-formula mismatches 0 | zero iff balanced mismatches 0 | 2D/n violations 0 | recurrent-point bound violations 0


n=5:  21,875 maps,  3,986 tables | minimum-formula mismatches 0 | zero iff balanced mismatches 0 | 2D/n violations 0 | recurrent-point bound violations 0


n=6: 513,216 maps, 59,486 tables | minimum-formula mismatches 0 | zero iff balanced mismatches 0 | 2D/n violations 0 | recurrent-point bound violations 0
[5s]
Landauer bound at 300 K: 2.871e-21 J per bit


## Section 8: hex-driven transformations and their profiles

In [8]:
def sha_stream(seed, n_hex):
    out, h = [], hashlib.sha256(seed).digest()
    while len(out) * 64 < n_hex:
        out.append(h.hex()); h = hashlib.sha256(h).digest()
    return "".join(out)[:n_hex]

def pi_hex(n_hex):
    bits = 4 * (n_hex + 16); one = 1 << bits
    def arctan_inv(x):
        total = term = one // x; x2, k, sign = x * x, 1, -1
        while term:
            term //= x2; total += sign * (term // (2 * k + 1)); sign, k = -sign, k + 1
        return total
    frac = 16 * arctan_inv(5) - 4 * arctan_inv(239) - (3 << bits)
    return format(frac >> (4 * 16), "x").zfill(n_hex)[:n_hex]

def permutation_from_hex(hexstr, n):
    p = list(range(n)); stream = bytes.fromhex(hexstr[: len(hexstr) // 2 * 2]); pos = 0
    for i in range(n - 1, 0, -1):
        mask = (1 << i.bit_length()) - 1
        while True:
            assert pos < len(stream), "hex stream too short"
            r = stream[pos] & mask; pos += 1
            if r <= i:
                break
        p[i], p[r] = p[r], p[i]
    return tuple(p)

assert pi_hex(32) == "243f6a8885a308d313198a2e03707344"
assert hashlib.sha256(b"").hexdigest().startswith("e3b0c442")
N = 128; total = log2_factorial(N)
for name, p in [("pi", permutation_from_hex(pi_hex(1200), N)), ("SHA-256 chain from empty input", permutation_from_hex(sha_stream(b"", 1200), N))]:
    print(f"\n{name}: {cycle_count(p)} cycles")
    print(" cuts   m   seen bits  hidden bits  distinguishes inverse  certified loops")
    for k in range(8):
        m = 2 ** k; T = table(p, m); hid = log2_fiber(T)
        print(f" {k:>4} {m:>4} {total - hid:10.2f} {hid:12.2f} {str(table(inverse(p), m) != T):>22} {components(T):>16}")


pi: 3 cycles
 cuts   m   seen bits  hidden bits  distinguishes inverse  certified loops
    0    1       0.00       716.16                  False                1
    1    2       3.19       712.97                  False                1
    2    4      25.96       690.21                   True                1
    3    8     104.05       612.11                   True                1
    4   16     250.34       465.82                   True                1
    5   32     427.72       288.44                   True                1
    6   64     589.16       127.00                   True                1
    7  128     716.16         0.00                   True                3

SHA-256 chain from empty input: 9 cycles
 cuts   m   seen bits  hidden bits  distinguishes inverse  certified loops
    0    1       0.00       716.16                  False                1
    1    2       4.26       711.90                  False                1
    2    4      22.36       693.80          

## Appendix A: the exact profile $I(T;C_m)$ for $N = 16,\\dots,1024$

In [9]:
for N in (16, 32, 64, 128, 256, 512, 1024):
    row = [f"{profile_formula(N, 2 ** k):.2f}" for k in range(N.bit_length()) if 2 ** k <= N]
    half = log2_factorial(N) / 2
    k_half = next(k for k in range(N.bit_length()) if profile_formula(N, 2 ** k) >= half)
    print(f"N={N:>5}: log2 N! = {log2_factorial(N):9.2f} | profile by cuts 0..{N.bit_length() - 1}: {', '.join(row)} | half reached at cut {k_half}")

N=   16: log2 N! =     44.25 | profile by cuts 0..4: 0.00, 2.09, 12.18, 28.78, 44.25 | half reached at cut 3
N=   32: log2 N! =    117.66 | profile by cuts 0..5: 0.00, 2.57, 16.86, 48.86, 86.18, 117.66 | half reached at cut 4
N=   64: log2 N! =    296.00 | profile by cuts 0..6: 0.00, 3.06, 21.43, 74.21, 153.81, 232.50, 296.00 | half reached at cut 4
N=  128: log2 N! =    716.16 | profile by cuts 0..7: 0.00, 3.55, 25.94, 101.33, 250.33, 427.24, 588.67, 716.16 | half reached at cut 5
N=  256: log2 N! =   1684.00 | profile by cuts 0..8: 0.00, 4.05, 30.43, 127.39, 369.98, 728.97, 1101.63, 1428.50, 1684.00 | half reached at cut 6
N=  512: log2 N! =   3875.17 | profile by cuts 0..9: 0.00, 4.55, 34.93, 152.42, 497.38, 1150.17, 1941.18, 2705.92, 3363.67, 3875.17 | half reached at cut 6
N= 1024: log2 N! =   8769.01 | profile by cuts 0..10: 0.00, 5.05, 39.43, 177.13, 619.31, 1667.37, 3214.68, 4876.81, 6426.01, 7745.51, 8769.01 | half reached at cut 7
